## XBRL US API - FERC schedule by entity  

### Authenticate for access token 
Click in the gray code cell below, then click the Run button above to execute the cell. Type your XBRL US Web account email, account password, Client ID, and secret as noted, pressing the Enter key on the keyboard after each entry.

XBRL US limits records returned for a query to improve efficiency; this script loops to collect all data from the Public Filings Database for a query. **Non-members might not be able to return all data for a query** - join XBRL US for comprehensive access - https://xbrl.us/join.

In [ ]:
import os, re, sys, json
import requests
import pandas as pd
from IPython.display import display, HTML
import numpy as np
import getpass
from datetime import datetime
import urllib
from urllib.parse import urlencode


class tokenInfoClass:
    access_token = None
    refresh_token = None
    email = None
    username = None
    client_id = None
    client_secret = None
    url = 'https://api.xbrl.us/oauth2/token'
    headers = {"Content-Type": "application/x-www-form-urlencoded"}
	
def refresh(info):
    refresh_auth = {
                'client_id': info.client_id, 
				'client_secret' : info.client_secret, 
				'grant_type' : 'refresh_token', 
				'platform' : 'ipynb', 
				'refresh_token' : info.refresh_token 
                }
    refreshres = requests.post(info.url, data=refresh_auth, headers=info.headers)
    refresh_json = refreshres.json()
    info.access_token = refresh_json['access_token']
    info.refresh_token = refresh_json['refresh_token']
    print('Your access token(%s) is refreshed for 60 minutes. If it expires again, run this cell to generate a new token and continue to use the query cells below.' % (info.access_token))
    return info	

tokenInfo = tokenInfoClass()

tokenInfo.email = input('Enter your XBRL US Web account email: ')
tokenInfo.password = getpass.getpass(prompt='Password: ')
tokenInfo.client_id = getpass.getpass(prompt='Client ID: ')
tokenInfo.client_secret = getpass.getpass(prompt='Secret: ')

body_auth = {'username' : tokenInfo.email, 
            'client_id': tokenInfo.client_id, 
            'client_secret' : tokenInfo.client_secret, 
            'password' : tokenInfo.password, 
            'grant_type' : 'password', 
            'platform' : 'ipynb' }

#print(body_auth)

payload = urlencode(body_auth)
res = requests.request("POST", tokenInfo.url, data=payload, headers=tokenInfo.headers)
auth_json = res.json()

if 'error' in auth_json:
    print("\n\nThere was a problem generating the access token: %s.  Run the first cell again and enter the credentials." % (auth_json['error_description']))
else:
    tokenInfo.access_token = auth_json['access_token']
    tokenInfo.refresh_token = auth_json['refresh_token']
    print ("\n\nYour access token expires in 60 minutes. After it expires, it should be regenerated automatically.  If not, run the cell rerun the first query cell. \n\nFor now, skip ahead to the section 'Make a Query'.")
	
#print(vars(tokenInfo))
print('\n\naccess token: ' + tokenInfo.access_token + ' refresh token: ' + tokenInfo.refresh_token)

### Make a query 
After the access token confirmation appears above, you can modify the query below and use the **_Cell >> Run_** menu option with the cell **immediately below this text** to run the query for updated results. 

The sample results are from 10+ years of data for companies filing data on the _Summary of Utility Plant and Accumulated Provisions for Depreciation, Amortization and Depletion_ , and may take several minutes to recreate.  

Modify the **_XBRL\_Elements_** to return data from a different FERC Form 1 Schedule, and/or change the **_entity\_codes_** to shorten the list. You can also change the base taxonomy corresponding to the Form by year or Form type. The queries to get lists are in the commented lines above each variable.
  
Refer to XBRL API documentation at https://xbrlus.github.io/xbrl-api/#/Facts/getFactDetails for other endpoints and parameters to filter and return. 

In [ ]:
# Define the parameters for the filter and fields to be returned, 
# run the loop to return results

endpoint = 'cube'

# Define the parameters of the query

# query all taxonomies published for a specific year
# https://api.xbrl.us/api/v1/dts/search?fields=dts.id,dts.version.sort(DESC),dts.taxonomy-name.sort(DESC)&dts.version=2023

# query unique Statements in the 2023 FERC Form 1 Taxonomy
# https://api.xbrl.us/api/v1/dts/730705/network/search?network.link-name=presentationLink&fields=network.role-description.sort(ASC),dts.id&unique

XBRL_Elements = ["200 - Schedule - Summary of Utility Plant and Accumulated Provisions for Depreciation, Amortization and Depletion"]

# query for a list of Form 1 filer entity codes:
# https://api.xbrl.us/api/v1/report/search?report.document-type=1&fields=report.entity-name.sort(ASC),entity.code&unique
entity_codes = [
'C000533','C001111' 
]

# Define data fields to return (multi-sort based on order)

fields = [ # this is the list of the characteristics of the data being returned by the query
		'period.fiscal-year.sort(DESC)',
		'period.fiscal-period.sort(DESC)',
		'entity.code',
		'report.entity-name',
		'report.id',
		'cube.description.sort(ASC)',
		'cube.tree-sequence.sort(ASC)',
		'cube.primary-local-name',
		'fact.value',
		'unit',
		'dimensions.count',
		'dimension-pair.sort(ASC)'
        ]

params = { # this is the list of what's being queried against the search endpoint
         'cube.description': ','.join(XBRL_Elements),  
         'entity.code': ','.join(entity_codes),
         # uncomment this line to run by report.id AND comment the prior line for entity.code 'report.id': ','.join(full_list),
         'fields': ','.join(fields)
         }

# Execute the query with loop for all results

# Set unique as TRUE or FALSE where TRUE drops any duplicate rows (ie. all fields identical)
unique = True

# Limit the number of rows displayed by the notebook (does not impact the data frame)
rows_to_display = 10 # Set as '' to display all rows in the notebook

### Execute the query with loop for all results 
### THIS SECTION DOES NOT NEED TO BE EDITED

search_endpoint = 'https://api.xbrl.us/api/v1/' + endpoint + '/search'
if unique:
    search_endpoint += "?unique"
orig_fields = params['fields']
offset_value = 0
res_df = []
count = 0
query_start = datetime.now()
printed = False
run_query = True

while True:
    if not printed:
        print("On", query_start.strftime("%c"), tokenInfo.email, "(client ID:", str(tokenInfo.client_id.split('-')[0]), "...) started the query and")
        printed = True
    retry = 0
    while retry < 3:
        res = requests.get(search_endpoint, params=params, headers={'Authorization' : 'Bearer {}'.format(tokenInfo.access_token)})
        res_json = res.json()
        if 'error' in res_json:
            if res_json['error_description'] == 'Bad or expired token':
                tokenInfo = refresh(tokenInfo)
            else: 
                print('There was an error: {}'.format(res_json['error_description']))
                run_query = False
                break
        else: 
		        break
        retry +=1
        if retry >= 3:
            print("Can't refresh the access token.  Run the first query block, then rerun the query.")
            run_query = False

    if not run_query:
       break

    print("up to", str(offset_value + res_json['paging']['limit']), "records are found so far ...")

    res_df += res_json['data']

    if res_json['paging']['count'] < res_json['paging']['limit']:
        print(" - this set contained fewer than the", res_json['paging']['limit'], "possible, only", str(res_json['paging']['count']), "records.")
        break
    else: 
        offset_value += res_json['paging']['limit'] 
        if 100 == res_json['paging']['limit']:
                params['fields'] = orig_fields + ',' + endpoint + '.offset({})'.format(offset_value)
                if offset_value == 10 * res_json['paging']['limit']:
                        break 
        elif 500 == res_json['paging']['limit']:
                params['fields'] = orig_fields + ',' + endpoint + '.offset({})'.format(offset_value)
                if offset_value == 4 * res_json['paging']['limit']:
                        break 
        params['fields'] = orig_fields + ',' + endpoint + '.offset({})'.format(offset_value)

if not 'error' in res_json:
    current_datetime = datetime.now().replace(microsecond=0)
    time_taken = current_datetime - query_start
    index = pd.DataFrame(res_df).index
    total_rows = len(index)
    your_limit = res_json['paging']['limit']
    limit_message = "If the results below match the limit noted above, you might not be seeing all rows, and should consider upgrading (https://xbrl.us/access-token).\n"
    
    if your_limit == 100:
        print("\nThis non-Member account has a limit of " , 10 * your_limit, " rows per query from our Public Filings Database. " + limit_message)
    elif your_limit == 500:
        print("\nThis Basic Individual Member account has a limit of ", 4 * your_limit, " rows per query from our Public Filings Database. " + limit_message)
    
    print("\nAt " + current_datetime.strftime("%c") +  ", the query finished with  ", str(total_rows), "  rows returned in " + str(time_taken) + " for \n" +  urllib.parse.unquote(res.url))
    
    df = pd.DataFrame(res_df)
    # the format truncates the HTML display of numerical values to two decimals; .csv data is unaffected
    pd.options.display.float_format = '{:,.2f}'.format
    display(HTML(df.to_html(max_rows=rows_to_display)))

On Mon Nov 18 16:45:12 2024 info@xbrl.us (client ID: 69e1257c ...) started the query and
up to 5000 records are found so far ...
 - this set contained fewer than the 5000 possible, only 4173 records.

At Mon Nov 18 16:45:28 2024, the query finished with   4173   rows returned in 0:00:15.380693 for 
https://api.xbrl.us/api/v1/cube/search?unique&cube.description=200+-+Schedule+-+Summary+of+Utility+Plant+and+Accumulated+Provisions+for+Depreciation,+Amortization+and+Depletion&entity.code=C000533,C001111&fields=period.fiscal-year.sort(DESC),period.fiscal-period.sort(DESC),entity.code,report.entity-name,report.id,cube.description.sort(ASC),cube.tree-sequence.sort(ASC),cube.primary-local-name,fact.value,unit,dimensions.count,dimension-pair.sort(ASC)


,period.fiscal-year,period.fiscal-period,entity.code,report.entity-name,report.id,cube.description,cube.tree-sequence,cube.primary-local-name,fact.value,unit,dimensions.count,dimension-pair
0,2024,2Q,C000533,Kentucky Power Company,753929,"200 - Schedule - Summary of Utility Plant and Accumulated Provisions for Depreciation, Amortization and Depletion",3,UtilityPlantInServiceClassified,"3,285,598,514.00",USD,0,
1,2024,2Q,C000533,Kentucky Power Company,753929,"200 - Schedule - Summary of Utility Plant and Accumulated Provisions for Depreciation, Amortization and Depletion",3,UtilityPlantInServiceClassified,"3,285,598,514.00",USD,1,[{'UtilityTypeAxis': 'ElectricUtilityMember'}]
2,2024,2Q,C001111,Baltimore Gas and Electric Company,753841,"200 - Schedule - Summary of Utility Plant and Accumulated Provisions for Depreciation, Amortization and Depletion",3,UtilityPlantInServiceClassified,"10,068,795,285.00",USD,1,[{'UtilityTypeAxis': 'ElectricUtilityMember'}]
3,2024,2Q,C001111,Baltimore Gas and Electric Company,753841,"200 - Schedule - Summary of Utility Plant and Accumulated Provisions for Depreciation, Amortization and Depletion",3,UtilityPlantInServiceClassified,"1,244,861,502.00",USD,1,[{'UtilityTypeAxis': 'CommonUtilityMember'}]
4,2024,2Q,C001111,Baltimore Gas and Electric Company,753841,"200 - Schedule - Summary of Utility Plant and Accumulated Provisions for Depreciation, Amortization and Depletion",3,UtilityPlantInServiceClassified,"15,404,286,687.00",USD,0,
...,...,...,...,...,...,...,...,...,...,...,...,...
4168,2010,Y,C001111,Baltimore Gas and Electric Company,418249,"200 - Schedule - Summary of Utility Plant and Accumulated Provisions for Depreciation, Amortization and Depletion",16,UtilityPlantNet,"4,541,951,837.00",USD,0,
4169,2010,Y,C000533,Kentucky Power Company,420966,"200 - Schedule - Summary of Utility Plant and Accumulated Provisions for Depreciation, Amortization and Depletion",34,AccumulatedProvisionForDepreciationAmortizationAndDepletionOfPlantUtility,"568,441,518.00",USD,0,
4170,2010,Y,C000533,Kentucky Power Company,420967,"200 - Schedule - Summary of Utility Plant and Accumulated Provisions for Depreciation, Amortization and Depletion",34,AccumulatedProvisionForDepreciationAmortizationAndDepletionOfPlantUtility,"568,441,518.00",USD,0,
4171,2010,Y,C001111,Baltimore Gas and Electric Company,418248,"200 - Schedule - Summary of Utility Plant and Accumulated Provisions for Depreciation, Amortization and Depletion",34,AccumulatedProvisionForDepreciationAmortizationAndDepletionOfPlantUtility,"2,659,775,623.00",USD,0,


In [ ]:
# If you run this program locally, you can save the output to a file on your computer (modify D:\results.csv to your system)
df.to_csv(r"D:\results.csv",sep=",")

# Google Colab users - comment out the line above and uncomment the code below to save the data frame as a .csv in your Google Drive

#from google.colab import drive
#drive.mount('drive')
#df.to_csv('data.csv')
#!cp data.csv "drive/My Drive/"